# Installation

In [ ]:
#pip uninstall -y transformers torch torchvision

In [ ]:
#!pip install git+https://github.com/dnth/rag-datakit.git

In [3]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import BatchSamplers
from datasets import load_dataset

In [4]:
# dataset = load_dataset("frankwong2001/ssf-train-valid-full-synthetic-batch10")
# dataset = load_dataset("frankwong2001/ssf-train-valid-full-synthetic-v2")
dataset = load_dataset("frankwong2001/ssf-train-valid-full-synthetic-v3")
dataset

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 3016
    })
    valid: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 754
    })
})

In [5]:
dataset['valid'][0]

{'anchor': 'The Venue Operations Manager is responsible for overseeing the running of venue operations, including the logistics requirements. He/She works closely with event services department to ensure client requirements are fulfilled in compliance to local health and safety standards. He reviews event plans to ensure generation of maximum yield for organisation. Meticulous and resourceful, he possesses excellent problem-solving skills and is able to react quickly to deviations in the project plans. He is able to work in a flexible workweek, including weekends, evenings, and public holidays, and is comfortable working in both an indoor and outdoor environment depending on the nature and requirements of the events.',
 'positive': 'The Venue Operations Manager is tasked with managing the day-to-day functions of venue operations, including logistical planning and execution. Collaborating closely with the event services team, he/she ensures that client needs are met while adhering to lo

# W&B and Model Configuration

In [6]:
import wandb
import os
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Fetch the WANDB_API_KEY from the environment
wandb_api_key = os.getenv("WAB_API_KEY")

# Log in using the API key
wandb.login(key=wandb_api_key)

model_id = "nomic-ai/modernbert-embed-base"
save_model_path = "./models/nomic-ai/modernbert-embed-base"
wandb.init(project="rag-datakit-finetunes", name="3_nomic-ai/modernbert-embed-base~frankwong2001/ssf-train-valid-full-synthetic-v2")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: frankwong2001 (frankwong2001-cxsanalytics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Training Arguments

In [7]:
args = SentenceTransformerTrainingArguments(
    output_dir=save_model_path,
    num_train_epochs=5,                         # number of epochs
    per_device_train_batch_size=32,             # train batch size
    gradient_accumulation_steps=16,             # for a global batch size of 512
    per_device_eval_batch_size=16,              # evaluation batch size
    warmup_ratio=0.1,                           # warmup ratio
    learning_rate=2e-5,                         # learning rate, 2e-5 is a good value
    lr_scheduler_type="cosine",                 # use cosine learning rate scheduler
    optim="adamw_torch_fused",
    tf32=False,                                 # use tf32 precision
    bf16=True,         
    #fp16=True,                                                  # use bf16 precision
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
    eval_strategy="epoch",                      # evaluate after each epoch
    save_strategy="epoch",                      # save after each epoch
    logging_strategy="epoch",                   # log after each epoch
    save_total_limit=3,                         # save only the last 3 models
    load_best_model_at_end=True,                # load the best model when training ends
    report_to="wandb",
    # gradient_checkpointing=True,              # use fused adamw optimizer
    #use_cache=False                            # disable the use of cache
    # weight_decay=0.01,                             # apply weight decay
    # max_grad_norm=0.5,                           # clip the gradient norm
    # warmup_steps=1500                           # number of warmup steps
    )

In [8]:
model = SentenceTransformer(model_id)
train_loss = MultipleNegativesRankingLoss(model)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['valid'],  
    loss=train_loss,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

# Execute Training


In [9]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.218400,0.012371
2,0.006600,0.001004
3,0.003100,0.000635
4,0.002700,0.000552
5,0.002000,0.000538


TrainOutput(global_step=30, training_loss=0.046537580185880265, metrics={'train_runtime': 194.9745, 'train_samples_per_second': 77.343, 'train_steps_per_second': 0.154, 'total_flos': 0.0, 'train_loss': 0.046537580185880265, 'epoch': 5.0})

#  Save & Upload Model

In [10]:
trainer.save_model()

In [11]:
import os
wandb.save(os.path.join(save_model_path, "*"))

wandb: WARNING Symlinked 15 files into the W&B run directory, call wandb.save again to sync new files.


['/root/rag-datakit/nbs-frank/wandb/run-20250911_110107-vixoy343/files/models/nomic-ai/modernbert-embed-base/2_Normalize',
 '/root/rag-datakit/nbs-frank/wandb/run-20250911_110107-vixoy343/files/models/nomic-ai/modernbert-embed-base/tokenizer_config.json',
 '/root/rag-datakit/nbs-frank/wandb/run-20250911_110107-vixoy343/files/models/nomic-ai/modernbert-embed-base/modules.json',
 '/root/rag-datakit/nbs-frank/wandb/run-20250911_110107-vixoy343/files/models/nomic-ai/modernbert-embed-base/config.json',
 '/root/rag-datakit/nbs-frank/wandb/run-20250911_110107-vixoy343/files/models/nomic-ai/modernbert-embed-base/model.safetensors',
 '/root/rag-datakit/nbs-frank/wandb/run-20250911_110107-vixoy343/files/models/nomic-ai/modernbert-embed-base/sentence_bert_config.json',
 '/root/rag-datakit/nbs-frank/wandb/run-20250911_110107-vixoy343/files/models/nomic-ai/modernbert-embed-base/tokenizer.json',
 '/root/rag-datakit/nbs-frank/wandb/run-20250911_110107-vixoy343/files/models/nomic-ai/modernbert-embed-b

In [12]:
wandb.finish()

eval/loss,█▁▁▁▁
eval/runtime,█▁▁▂▁
eval/samples_per_second,▁█▇▇▇
eval/steps_per_second,▁█▇▇▇
train/epoch,▁▁▃▃▅▅▆▆███
train/global_step,▁▁▃▃▅▅▆▆███
train/grad_norm,█▁▁▁▁
train/learning_rate,█▇▄▂▁
train/loss,█▁▁▁▁
eval/loss,0.00054
eval/runtime,3.0482


# Push to Hugging Face

In [13]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import Trainer

# Load environment variables from .env file
load_dotenv()

# Fetch the Hugging Face API key from the environment
hf_api_key = os.getenv("HF_TOKEN")

# Log in using the Hugging Face API key
login(token=hf_api_key)

# Assuming you have a Trainer object `trainer`
trainer.model.push_to_hub("frankwong2001/3_modernbert-embed-base", exist_ok=True)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmph_v0q7c9/model.safetensors    :   0%|          |  552kB /  596MB            

'https://huggingface.co/frankwong2001/3_modernbert-embed-base/commit/da1879219ceaa2112c4a9a3ac9bd6482b6269708'